In [ ]:
import pandas as pd
import numpy as np

input_path = "it_support_team_performance.csv"
df = pd.read_csv(input_path)

print("Initial shape:", df.shape)


df = df.drop_duplicates(subset="Ticket_ID", keep="first")
print("After removing duplicates:", df.shape)


df["SLA_Met"] = (
    df["SLA_Met"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "yes": "Yes",
        "y": "Yes",
        "true": "Yes",
        "no": "No",
        "n": "No",
        "false": "No"
    })
)


df["Issue_Type"] = (
    df["Issue_Type"]
    .astype(str)
    .str.strip()
    .replace({
        "Hardwre": "Hardware"
    })
)

numeric_cols = [
    "Tickets_Assigned",
    "Tickets_Resolved",
    "Avg_Resolution_Time_hrs",
    "SLA_Target_hrs",
    "Customer_Satisfaction",
    "Reopened_Tickets"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")


df.loc[df["Avg_Resolution_Time_hrs"] <= 0, "Avg_Resolution_Time_hrs"] = np.nan


df["Avg_Resolution_Time_hrs"] = (
    df.groupby("Priority")["Avg_Resolution_Time_hrs"]
      .transform(lambda x: x.fillna(x.median()))
)


df["Customer_Satisfaction"] = (
    df.groupby("Agent_ID")["Customer_Satisfaction"]
      .transform(lambda x: x.fillna(x.median()))
)


df["Customer_Satisfaction"] = df["Customer_Satisfaction"].fillna(
    df["Customer_Satisfaction"].median()
)


df["Tickets_Resolved"] = np.minimum(
    df["Tickets_Resolved"],
    df["Tickets_Assigned"]
)


df["Reopened_Tickets"] = np.minimum(
    df["Reopened_Tickets"],
    df["Tickets_Resolved"]
)


df["SLA_Met"] = np.where(
    (df["Avg_Resolution_Time_hrs"] <= df["SLA_Target_hrs"]) &
    (df["Tickets_Resolved"] > 0),
    "Yes",
    "No"
)

df["Resolution_Efficiency"] = (
    df["Tickets_Resolved"] / df["Tickets_Assigned"]
).round(2)


df["First_Time_Resolution"] = np.where(
    df["Reopened_Tickets"] == 0,
    "Yes",
    "No"
)


df["SLA_Breach_Flag"] = np.where(
    df["SLA_Met"] == "No",
    "Yes",
    "No"
)


assert df["Ticket_ID"].is_unique, "Duplicate Ticket_IDs still exist"
assert (df["Avg_Resolution_Time_hrs"] > 0).all(), "Invalid resolution times exist"
assert (df["Tickets_Resolved"] <= df["Tickets_Assigned"]).all(), "Resolved > Assigned"
assert (df["Reopened_Tickets"] <= df["Tickets_Resolved"]).all(), "Reopened > Resolved"

print("Data validation passed.")


output_path = "it_support_team_performance_clean.csv"
df.to_csv(output_path, index=False)

print("Clean file saved as:", output_path)
df

Initial shape: (26, 11)
After removing duplicates: (25, 11)
Data validation passed.
Clean file saved as: it_support_team_performance_clean.csv


,Ticket_ID,Agent_ID,Priority,Issue_Type,Tickets_Assigned,Tickets_Resolved,Avg_Resolution_Time_hrs,SLA_Target_hrs,SLA_Met,Customer_Satisfaction,Reopened_Tickets,Resolution_Efficiency,First_Time_Resolution,SLA_Breach_Flag
0,T001,A101,High,Network,18,15,6.50,6,No,3.0,2,0.83,No,Yes
1,T002,A102,Medium,Software,14,14,4.20,6,Yes,4.0,0,1.00,Yes,No
2,T003,A103,Critical,Security,10,7,9.10,4,No,2.0,3,0.70,No,Yes
3,T004,A101,Low,Hardware,20,19,3.10,8,Yes,5.0,0,0.95,Yes,No
4,T005,A104,High,Database,16,12,7.80,6,No,3.0,1,0.75,No,Yes
5,T006,A102,Medium,Software,15,15,3.90,6,Yes,4.0,0,1.00,Yes,No
6,T007,A105,High,Network,12,10,6.20,6,No,3.0,1,0.83,No,Yes
7,T008,A103,Critical,Security,9,6,10.40,4,No,1.0,4,0.67,No,Yes
8,T009,A104,Medium,Hardware,17,16,4.30,8,Yes,4.0,0,0.94,Yes,No
9,T010,A101,High,Database,19,17,6.00,6,Yes,4.0,1,0.89,No,No
